## Project Description

For the Words to Embeddings group work assignment, we decided to create word embeddings on a small-, medium-, and large-sized dataset that builds off the same corpus of text and then test to see how relationships between words differ between the sets of vectors. The corpus of text we used is The Complete Works of Shakespeare pulled from Project Gutenberg. The small dataset consists of the first 25% of the plain text file, the medium dataset consists of the first 50% of the file, and the large dataset is the entire file.

After all of the embeddings were created and trained with each sized dataset, we tested to see how certain relationships between words differed. The testing consisted of using the vector offset method to get the word embedding for "queen" from the expression "king - man + woman" and seeing how off each set of embeddings was from the true vector for "queen". 

## Results

In [ ]:
from gensim.utils import simple_preprocess
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize
from gensim.models import Word2Vec

text = Path("Shakespeare.txt").read_text(encoding='utf-8', errors='ignore')
# simple_preprocess returns tokens (lowercased), removes tokens <2 chars by default
word_tokens = simple_preprocess(text, deacc=True)  # deacc=True removes accents/punctuation

In [ ]:
def create_sentence_tokens(text):
    sentences = sent_tokenize(text) 
    sentence_tokens = [simple_preprocess(sent) for sent in sentences]
    return sentence_tokens

def create_model(title, sentences):
    model = Word2Vec(
        sentences=sentences,
        vector_size=200,   # embedding dimension
        window=5,
        min_count=2,       # ignore rare words (adjust to your corpus size)
        sg=1,              # 1 = skip-gram, 0 = CBOW
        negative=10,
        sample=1e-4,
        epochs=5
    )

    model.save(f'{title}.model')

In [50]:
model = Word2Vec.load("word2vec.model")

import numpy as np

vec = model.wv['lord'] - model.wv['man'] + model.wv['woman']

# find closest word
similar = model.wv.similar_by_vector(vec, topn=5)
print(similar)


[('lord', 0.8350991606712341), ('katherine', 0.6984719038009644), ('lady', 0.690411388874054), ('ophelia', 0.6827850341796875), ('adieu', 0.6805949211120605)]


In [27]:
# vector for a word
print(model.wv['king'].shape)        # (200,)

# most similar
print(model.wv.most_similar('king', topn=10))


(200,)
[('prince', 0.963988184928894), ('clarence', 0.9594987630844116), ('edward', 0.9581371545791626), ('york', 0.9532898664474487), ('henry', 0.950258195400238), ('richard', 0.9445779919624329), ('duke', 0.9348256587982178), ('harry', 0.9288215041160583), ('warwick', 0.9245917797088623), ('cardinal', 0.9239512085914612)]


In [15]:
model.wv.save_word2vec_format("vectors.txt", binary=False)


In [26]:
result = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)
print(result)
# [('queen', 0.72), ('princess', 0.68), ...]


[('prince', 0.9508436322212219), ('edward', 0.9456694722175598), ('clarence', 0.9425180554389954), ('richard', 0.9258468151092529), ('henry', 0.9240415692329407)]


In [ ]:
text = Path("DonQuixote.txt").read_text(encoding='utf-8', errors='ignore')

sentence_tokens = create_sentence_tokens(text)

create_model("DonQuixote", sentence_tokens)